# EII — Closing Analyses (Pre-writing)

Three analyses required to complete the dataset before paper writing begins.
All use data already produced in Phases 1–3.

**Analysis A — EII × Area correlation (Section 4.1)**
Pearson and Spearman correlation between EII and Area per year.
Proportion of variance in EII not explained by Area (1 - R²).
Residual distribution: cells where EII deviates most from Area.

**Analysis B — 5×5 frequency matrices for snapshot years (Section 4.3)**
Area × EII joint distribution for anchor years: 1986, 1995, 2004, 2012, 2020, 2023.
Difference matrices between consecutive periods.

**Analysis C — Temporal interval sensitivity (Section 4.7)**
Comparison of 5-year vs. 10-year snapshot matrices.
Maximum absolute difference in cell frequencies between interval choices.

**Effective analysis period:** 1986–2023 (38 years).

---
## 1. Configuration — edit only this cell

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Annual EII dataset from Phase 2
EII_CSV  = r"D:\Modelo_LAPIG\phase2_annual\eii_HEX20_annual.csv"

# Annual Area dataset from Phase 2
AREA_CSV = r"D:\Modelo_LAPIG\phase2_annual\area_HEX20_annual.csv"

# Output folder
OUTPUT_FOLDER = r"D:\Modelo_LAPIG\phase3_closing"

# Effective analysis period
YEAR_START = 1986
YEAR_END   = 2023

# Divergence threshold for state classification (provisional)
THRESHOLD = 0.5

# Snapshot anchor years for 5x5 matrices (Analysis B)
SNAPSHOT_YEARS = [1986, 1995, 2004, 2012, 2020, 2023]

# Bin edges for 5x5 matrix (0-20%, 20-40%, ..., 80-100%)
BIN_EDGES  = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
BIN_LABELS = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]

---
## 2. Dependencies and data loading

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

years = list(range(YEAR_START, YEAR_END + 1))

# Load EII
df_eii = pd.read_csv(EII_CSV)
eii_cols = sorted([c for c in df_eii.columns
                   if any(str(y) in c for y in years) and "eii" in c.lower()],
                  key=lambda c: int(c[-4:]))
EII = df_eii[eii_cols].values.astype(float)  # (11500, 38)
ids = df_eii["ID_UNICO"].values

# Load Area
df_area = pd.read_csv(AREA_CSV)
area_cols = sorted([c for c in df_area.columns
                    if any(str(y) in c for y in years) and "area" in c.lower()],
                   key=lambda c: int(c[-4:]))

# If area columns use 'prop' naming instead of 'area'
if len(area_cols) == 0:
    area_cols = sorted([c for c in df_area.columns
                        if any(str(y) in c for y in years)],
                       key=lambda c: int(c[-4:]))

AREA = df_area[area_cols].values.astype(float)  # (11500, 38)

print(f"EII  matrix: {EII.shape}  ({EII.shape[0]:,} cells x {EII.shape[1]} years)")
print(f"Area matrix: {AREA.shape}  ({AREA.shape[0]:,} cells x {AREA.shape[1]} years)")
print(f"Years: {YEAR_START}-{YEAR_END}")
assert EII.shape == AREA.shape, "EII and Area matrices must have the same shape!"
print("Shape check: OK")

---
## Analysis A — EII × Area correlation (RQ1)

Pearson r and Spearman rho computed annually across all cells.
Key result: proportion of variance in EII not explained by Area (1 - R²).

In [ ]:
pearson_r   = np.full(len(years), np.nan)
spearman_r  = np.full(len(years), np.nan)
r_squared   = np.full(len(years), np.nan)
unexplained = np.full(len(years), np.nan)

for j, yr in enumerate(years):
    eii_j  = EII[:, j]
    area_j = AREA[:, j]
    mask   = np.isfinite(eii_j) & np.isfinite(area_j)
    if mask.sum() < 100:
        continue

    pr, _ = stats.pearsonr(area_j[mask], eii_j[mask])
    sr, _ = stats.spearmanr(area_j[mask], eii_j[mask])
    pearson_r[j]   = round(pr, 4)
    spearman_r[j]  = round(sr, 4)
    r_squared[j]   = round(pr**2, 4)
    unexplained[j] = round(1 - pr**2, 4)

df_corr = pd.DataFrame({
    "year":        years,
    "pearson_r":   pearson_r,
    "spearman_r":  spearman_r,
    "r_squared":   r_squared,
    "unexplained": unexplained,
})

# Save
corr_path = os.path.join(OUTPUT_FOLDER, "eii_area_correlation_annual.csv")
df_corr.to_csv(corr_path, index=False, encoding="utf-8-sig")

# Print summary
print("=== EII x Area Correlation ===")
print(f"\nPearson r:")
print(f"  {YEAR_START}: {df_corr.iloc[0]['pearson_r']:.4f}")
print(f"  {YEAR_END}:   {df_corr.iloc[-1]['pearson_r']:.4f}")
print(f"  Min: {df_corr['pearson_r'].min():.4f}  Max: {df_corr['pearson_r'].max():.4f}")
print(f"\nR-squared (variance explained by Area):")
print(f"  {YEAR_START}: {df_corr.iloc[0]['r_squared']:.4f} "
      f"({df_corr.iloc[0]['r_squared']*100:.1f}%)")
print(f"  {YEAR_END}:   {df_corr.iloc[-1]['r_squared']:.4f} "
      f"({df_corr.iloc[-1]['r_squared']*100:.1f}%)")
print(f"\nVariance in EII NOT explained by Area (1-R²):")
print(f"  {YEAR_START}: {df_corr.iloc[0]['unexplained']:.4f} "
      f"({df_corr.iloc[0]['unexplained']*100:.1f}%)")
print(f"  {YEAR_END}:   {df_corr.iloc[-1]['unexplained']:.4f} "
      f"({df_corr.iloc[-1]['unexplained']*100:.1f}%)")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(years, pearson_r,  label="Pearson r",  color="steelblue", linewidth=2)
axes[0].plot(years, spearman_r, label="Spearman rho", color="darkorange",
             linewidth=2, linestyle="--")
axes[0].set_ylim(0.8, 1.0)
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Correlation coefficient")
axes[0].set_title("Annual EII x Area correlation")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(years, unexplained, alpha=0.3, color="steelblue")
axes[1].plot(years, unexplained, color="steelblue", linewidth=2)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("1 - R² (proportion)")
axes[1].set_title("Variance in EII not explained by Area")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_FOLDER, "eii_area_correlation.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {corr_path}")
print(f"Saved: {fig_path}")

### A2 — Residual analysis

Cells where EII deviates most from Area-predicted values.
High residuals = cells where EII provides most additional information.

In [ ]:
# Compute residuals for first and last year
residual_summary = []

for yr_label, j in [(YEAR_START, 0), (YEAR_END, len(years)-1)]:
    eii_j  = EII[:, j]
    area_j = AREA[:, j]
    mask   = np.isfinite(eii_j) & np.isfinite(area_j)

    # OLS: EII = a + b * Area
    slope, intercept, r, p, se = stats.linregress(area_j[mask], eii_j[mask])
    predicted = intercept + slope * area_j
    residuals = eii_j - predicted

    residual_summary.append({
        "year":         yr_label,
        "slope":        round(slope, 4),
        "intercept":    round(intercept, 4),
        "r_squared":    round(r**2, 4),
        "resid_sd":     round(np.nanstd(residuals), 4),
        "pct_abs_gt05": round((np.abs(residuals[mask]) > 0.05).mean()*100, 2),
        "pct_abs_gt10": round((np.abs(residuals[mask]) > 0.10).mean()*100, 2),
    })

    print(f"\nResiduals {yr_label}:")
    print(f"  Slope: {slope:.4f}  Intercept: {intercept:.4f}")
    print(f"  Residual SD: {np.nanstd(residuals):.4f}")
    print(f"  |residual| > 0.05: {(np.abs(residuals[mask])>0.05).mean()*100:.1f}% of cells")
    print(f"  |residual| > 0.10: {(np.abs(residuals[mask])>0.10).mean()*100:.1f}% of cells")

pd.DataFrame(residual_summary).to_csv(
    os.path.join(OUTPUT_FOLDER, "eii_area_residuals_summary.csv"),
    index=False, encoding="utf-8-sig")
print("\nResidual summary saved.")

---
## Analysis B — 5×5 Area × EII frequency matrices (Section 4.3)

Joint distribution of Area and EII for each anchor year.
Difference matrices between consecutive periods.

In [ ]:
def make_matrix(eii_vec, area_vec, bin_edges):
    """Build 5x5 frequency matrix (rows=Area bins, cols=EII bins)."""
    mask  = np.isfinite(eii_vec) & np.isfinite(area_vec)
    n     = mask.sum()
    mat   = np.zeros((5, 5), dtype=int)
    eii_b = np.clip(np.digitize(eii_vec[mask],  bin_edges) - 1, 0, 4)
    are_b = np.clip(np.digitize(area_vec[mask], bin_edges) - 1, 0, 4)
    for i, j in zip(are_b, eii_b):
        mat[i, j] += 1
    return mat, n


matrices = {}  # year -> (mat_counts, mat_pct)

for yr in SNAPSHOT_YEARS:
    j      = years.index(yr)
    mat, n = make_matrix(EII[:, j], AREA[:, j], BIN_EDGES)
    pct    = np.round(mat / n * 100, 2)
    matrices[yr] = (mat, pct)

    # Save CSV
    pd.DataFrame(mat,  index=BIN_LABELS, columns=BIN_LABELS).to_csv(
        os.path.join(OUTPUT_FOLDER, f"matrix_counts_{yr}.csv"), encoding="utf-8-sig")
    pd.DataFrame(pct,  index=BIN_LABELS, columns=BIN_LABELS).to_csv(
        os.path.join(OUTPUT_FOLDER, f"matrix_pct_{yr}.csv"), encoding="utf-8-sig")

    # Print diagonal summary
    coupled = mat[0,0]+mat[1,1]+mat[2,2]+mat[3,3]+mat[4,4]
    type_i  = mat[3,0]+mat[3,1]+mat[4,0]+mat[4,1]  # high area, low EII
    type_ii = mat[0,3]+mat[0,4]+mat[1,3]+mat[1,4]  # low area, high EII
    print(f"{yr}: coupled={100*coupled/n:.1f}%  "
          f"Type I={100*type_i/n:.1f}%  Type II={100*type_ii/n:.1f}%")

# Difference matrices between consecutive periods
diff_records = []
for k in range(len(SNAPSHOT_YEARS)-1):
    yr1, yr2 = SNAPSHOT_YEARS[k], SNAPSHOT_YEARS[k+1]
    diff_pct = matrices[yr2][1] - matrices[yr1][1]
    pd.DataFrame(np.round(diff_pct, 2),
                 index=BIN_LABELS, columns=BIN_LABELS).to_csv(
        os.path.join(OUTPUT_FOLDER, f"matrix_diff_{yr1}_{yr2}.csv"),
        encoding="utf-8-sig")
    diff_records.append({"period": f"{yr1}-{yr2}",
                         "max_abs_diff": round(np.abs(diff_pct).max(), 2)})

print("\nMax absolute difference between consecutive matrix periods:")
for r in diff_records:
    print(f"  {r['period']}: {r['max_abs_diff']:.2f} pp")

print("\nAll matrices saved.")

### B2 — Plot matrices

In [ ]:
n_snap = len(SNAPSHOT_YEARS)
fig, axes = plt.subplots(1, n_snap, figsize=(4*n_snap, 4))

vmax = max(matrices[yr][1].max() for yr in SNAPSHOT_YEARS)

for ax, yr in zip(axes, SNAPSHOT_YEARS):
    pct = matrices[yr][1]
    im  = ax.imshow(pct, origin="lower", aspect="auto",
                    cmap="YlOrRd", vmin=0, vmax=vmax)
    ax.set_xticks(range(5))
    ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(5))
    ax.set_yticklabels(BIN_LABELS, fontsize=7)
    ax.set_title(str(yr), fontsize=10)
    ax.set_xlabel("EII", fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel("Area", fontsize=8)
    for i in range(5):
        for j in range(5):
            col = "white" if pct[i,j] > vmax*0.6 else "black"
            ax.text(j, i, f"{pct[i,j]:.1f}", ha="center", va="center",
                    fontsize=6, color=col)

plt.suptitle("Area x EII 5x5 frequency matrices (% of cells)", fontsize=11)
plt.tight_layout()
fig_path = os.path.join(OUTPUT_FOLDER, "matrices_snapshots.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

---
## Analysis C — Temporal interval sensitivity (Section 4.7)

Compares 5-year vs. 10-year snapshot intervals.
Tests whether the choice of snapshot frequency affects the
Area x EII distribution matrices materially.

In [ ]:
# Define snapshot sets
snaps_5yr  = [y for y in range(YEAR_START, YEAR_END+1, 5)]
snaps_10yr = [y for y in range(YEAR_START, YEAR_END+1, 10)]

# Ensure endpoints are included
if YEAR_END not in snaps_5yr:  snaps_5yr.append(YEAR_END)
if YEAR_END not in snaps_10yr: snaps_10yr.append(YEAR_END)

print(f"5-year snapshots:  {snaps_5yr}")
print(f"10-year snapshots: {snaps_10yr}")

# Build matrices for all snapshots
mats_5yr  = {}
mats_10yr = {}

for yr in set(snaps_5yr + snaps_10yr):
    j = years.index(yr)
    mat, n = make_matrix(EII[:, j], AREA[:, j], BIN_EDGES)
    pct = mat / n * 100
    if yr in snaps_5yr:  mats_5yr[yr]  = pct
    if yr in snaps_10yr: mats_10yr[yr] = pct

# For common years, compute difference between the two regimes
common_years = sorted(set(snaps_5yr) & set(snaps_10yr))

print("\nMax absolute difference at common snapshot years:")
max_diffs = []
for yr in common_years:
    # Compare the preceding interval under each scheme
    # Both schemes sample yr — compare the year-to-year trajectory
    max_diffs.append(np.abs(mats_5yr[yr] - mats_10yr[yr]).max())
    print(f"  {yr}: {max_diffs[-1]:.4f} pp (trivially 0 — same year, same matrix)")

# More meaningful comparison: for a fixed endpoint (2023),
# does the intermediate snapshot year matter?
# Compare matrix at 2023 with two preceding intervals
print("\nTrajectory comparison — from start to end via different intermediate years:")
mat_start = mats_5yr[YEAR_START]
mat_end   = mats_5yr[YEAR_END]
diff_total = mat_end - mat_start

# Midpoint under 5-year scheme vs. 10-year scheme
mid_5  = snaps_5yr[len(snaps_5yr)//2]
mid_10 = snaps_10yr[len(snaps_10yr)//2]
diff_at_mid = np.abs(mats_5yr.get(mid_5, mats_10yr[mid_10]) -
                     mats_10yr.get(mid_10, mats_5yr[mid_5])).max()

print(f"  Midpoint under 5yr scheme:  {mid_5}")
print(f"  Midpoint under 10yr scheme: {mid_10}")
print(f"  Max matrix difference at midpoints: {diff_at_mid:.2f} pp")

# Summary
summary_c = pd.DataFrame({
    "interval": ["5yr", "10yr"],
    "n_snapshots": [len(snaps_5yr), len(snaps_10yr)],
    "years": [str(snaps_5yr), str(snaps_10yr)],
    "max_diff_vs_other": [diff_at_mid, diff_at_mid],
})
summary_c.to_csv(
    os.path.join(OUTPUT_FOLDER, "temporal_sensitivity_summary.csv"),
    index=False, encoding="utf-8-sig")

# Plot: matrices at midpoints under each scheme
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (yr, mat_d, title) in zip(axes, [
    (YEAR_START, mat_start, f"Start ({YEAR_START})"),
    (mid_5, mats_5yr.get(mid_5, mats_10yr[mid_5]),
     f"Midpoint 5yr scheme ({mid_5})"),
    (YEAR_END, mat_end, f"End ({YEAR_END})"),
]):
    im = ax.imshow(mat_d, origin="lower", aspect="auto",
                   cmap="YlOrRd", vmin=0, vmax=vmax)
    ax.set_xticks(range(5))
    ax.set_xticklabels(BIN_LABELS, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(5))
    ax.set_yticklabels(BIN_LABELS, fontsize=7)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("EII", fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel("Area", fontsize=8)
    for i in range(5):
        for j in range(5):
            col = "white" if mat_d[i,j] > vmax*0.6 else "black"
            ax.text(j, i, f"{mat_d[i,j]:.1f}", ha="center", va="center",
                    fontsize=6, color=col)

plt.suptitle("Temporal interval sensitivity — Area x EII matrices", fontsize=11)
plt.tight_layout()
fig_path = os.path.join(OUTPUT_FOLDER, "temporal_sensitivity.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {fig_path}")

---
## Summary — all outputs produced

| File | Analysis | Paper section |
|---|---|---|
| `eii_area_correlation_annual.csv` | A — correlation | 4.1 |
| `eii_area_correlation.png` | A — correlation plot | 4.1 |
| `eii_area_residuals_summary.csv` | A — residuals | 4.1 |
| `matrix_counts_{year}.csv` | B — frequency matrix | 4.3 |
| `matrix_pct_{year}.csv` | B — frequency matrix % | 4.3 / Supp S1 |
| `matrix_diff_{yr1}_{yr2}.csv` | B — difference matrices | 4.3 |
| `matrices_snapshots.png` | B — visual overview | 4.3 |
| `temporal_sensitivity_summary.csv` | C — interval sensitivity | 4.7 / Supp S2 |
| `temporal_sensitivity.png` | C — sensitivity plot | 4.7 |

In [ ]:
import glob
files = sorted(glob.glob(os.path.join(OUTPUT_FOLDER, "*")))
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Files produced: {len(files)}")
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):50s} {size_kb:6.1f} KB")
print("\nAll closing analyses complete. Ready to begin paper writing.")